# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/20-PythonFlaskKullaniciGirisi.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 20 - Flask'ta Kullanıcı Girişi, Session ve Yetkilendirme

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bu derste Flask web uygulamamıza kullanıcı hesapları ekleyeceğiz.

Önceki derste:

- Flask,
- SQLite,
- HTML form,
- Jinja,
- CRUD

konularını birleştirmiştik.

Şimdi uygulamamıza:

- kullanıcı kaydı,
- güvenli parola saklama,
- giriş yapma,
- session,
- çıkış yapma,
- giriş gerektiren sayfalar,
- kullanıcı rolü,
- yetkilendirme

özelliklerini ekleyeceğiz.

> **Çalışma ortamı notu:** Flask uygulamalarını Visual Studio, VS Code veya yerel Python ortamında çalıştırmanız önerilir. Colab bu notebook'u ders dokümanı olarak incelemek için kullanılabilir.


# 1. Kimlik Doğrulama ve Yetkilendirme

Web uygulamalarında iki kavram birbirinden ayrılmalıdır.

### Kimlik Doğrulama

Kullanıcının kim olduğunu doğrulama işlemidir.

Örnek:

```text
Kullanıcı adı + parola → Giriş
```

### Yetkilendirme

Giriş yapmış kullanıcının hangi işlemleri yapabileceğini belirleme işlemidir.

Örnek:

```text
Öğrenci → yalnızca kendi profilini görüntüler
Öğretmen → öğrenci kayıtlarını düzenleyebilir
Yönetici → kullanıcıları yönetebilir
```

Kısaca:

**Authentication → Sen kimsin?**

**Authorization → Neleri yapabilirsin?**


# 2. Kullanıcı Giriş Sisteminin Akışı

```text
Kullanıcı
↓
Giriş Formu
↓
Kullanıcı Adı + Parola
↓
Veritabanında kullanıcıyı bul
↓
Parola hash kontrolü
↓
Doğruysa session oluştur
↓
Kullanıcı giriş yapmış kabul edilir
```

Çıkış yapıldığında session temizlenir.


# 3. Parolalar Neden Düz Metin Saklanmamalıdır?

Yanlış yaklaşım:

```text
KullaniciAdi | Parola
ali          | 123456
ayse         | python123
```

Veritabanı ele geçirilirse bütün parolalar doğrudan görülür.

Bu nedenle parola veritabanında **hash** biçiminde saklanmalıdır.


# 4. Hash Nedir?

Hash işlemi bir metni tek yönlü olarak farklı bir değere dönüştürür.

Örneğin mantıksal olarak:

```text
python123
↓
hash işlemi
↓
uzun ve okunması zor bir değer
```

Giriş sırasında kullanıcının yazdığı parola doğrudan veritabanındaki parola ile karşılaştırılmaz.

Bunun yerine uygun parola kontrol fonksiyonu kullanılır.


# 5. Werkzeug Parola Araçları

Flask ile birlikte kullanılan Werkzeug kütüphanesinde parola hash işlemleri için hazır güvenlik fonksiyonları vardır.


In [ ]:
from werkzeug.security import (
    generate_password_hash,
    check_password_hash
)


Kullanacağımız iki temel fonksiyon:

```python
generate_password_hash()
```

parolayı güvenli biçimde hashler.

```python
check_password_hash()
```

kullanıcının girdiği parolanın kayıtlı hash ile eşleşip eşleşmediğini kontrol eder.


# 6. Parola Hash Oluşturmak

In [ ]:
from werkzeug.security import generate_password_hash

parola = "Python123"

hash_degeri = generate_password_hash(parola)

print(hash_degeri)


Aynı paroladan üretilen hash değerinin görünümü her çalıştırmada aynı olmak zorunda değildir.

Uygulamada hash değerinin kendisini anlamaya veya elle karşılaştırmaya çalışmayız.


# 7. Parolayı Kontrol Etmek

In [ ]:
from werkzeug.security import (
    generate_password_hash,
    check_password_hash
)

hash_degeri = generate_password_hash("Python123")

print(
    check_password_hash(
        hash_degeri,
        "Python123"
    )
)

print(
    check_password_hash(
        hash_degeri,
        "YanlisParola"
    )
)


Sonuç:

- doğru parola → `True`
- yanlış parola → `False`

olur.


# 8. Kullanıcı Tablosu

SQLite veritabanımıza kullanıcı tablosu ekleyelim.

Örnek alanlar:

- Id
- KullaniciAdi
- ParolaHash
- Rol

SQL:

```sql
CREATE TABLE IF NOT EXISTS Kullanicilar (
    Id INTEGER PRIMARY KEY AUTOINCREMENT,
    KullaniciAdi TEXT UNIQUE NOT NULL,
    ParolaHash TEXT NOT NULL,
    Rol TEXT NOT NULL DEFAULT 'kullanici'
)
```


# 9. Kullanıcı Tablosunu Oluşturan Fonksiyon

In [ ]:
import sqlite3

DATABASE = "uygulama.db"

def kullanici_tablosu_hazirla():
    with sqlite3.connect(DATABASE) as db:
        db.execute("""
        CREATE TABLE IF NOT EXISTS Kullanicilar (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            KullaniciAdi TEXT UNIQUE NOT NULL,
            ParolaHash TEXT NOT NULL,
            Rol TEXT NOT NULL DEFAULT 'kullanici'
        )
        """)


# 10. Rol Alanı Neden Var?

Rol, kullanıcının yetkisini belirlemek için kullanılabilir.

Örneğin:

```text
kullanici
ogretmen
yonetici
```

Bu derste temel olarak:

```text
kullanici
yonetici
```

rollerini kullanacağız.


# 11. Flask Uygulamasında Gerekli İçe Aktarmalar

In [ ]:
from flask import (
    Flask,
    render_template,
    request,
    redirect,
    url_for,
    flash,
    session,
    g
)

import sqlite3
import functools

from werkzeug.security import (
    generate_password_hash,
    check_password_hash
)


# 12. Secret Key Neden Gerekir?

Flask session yapısı için uygulamanın gizli bir anahtara ihtiyacı vardır.

Bu anahtar session verisinin bütünlüğünün korunmasında kullanılır.

Secret key:

- uzun,
- rastgele,
- gizli

olmalıdır.

Kaynak kod deposuna gerçek üretim anahtarı yazılmamalıdır.


# 13. Rastgele Secret Key Üretmek

Terminalde:

```text
python -c "import secrets; print(secrets.token_hex())"
```

komutuyla güçlü bir rastgele değer üretilebilir.

Bu değer doğrudan GitHub'a yüklenmemelidir.


# 14. Ortam Değişkeninden Secret Key Almak

Örnek yaklaşım:


In [ ]:
import os
from flask import Flask

app = Flask(__name__)

app.config["SECRET_KEY"] = os.environ["FLASK_SECRET_KEY"]


Bu kodda `FLASK_SECRET_KEY` ortam değişkeni tanımlı değilse uygulama hata verir.

Bu davranış, uygulamanın yanlışlıkla zayıf bir varsayılan anahtarla başlamasını engelleyebilir.


# 15. Windows'ta Ortam Değişkeni

PowerShell örneği:

```text
$env:FLASK_SECRET_KEY="uzun-rastgele-gizli-deger"
```

Aynı terminal oturumundan Flask uygulaması çalıştırılabilir.


# 16. Session Nedir?

Session, kullanıcıya ait bazı bilgilerin birden fazla HTTP isteği boyunca hatırlanmasını sağlar.

Örneğin giriş yaptıktan sonra:

```python
session["user_id"] = 5
```

değeri tutulabilir.

Böylece sonraki sayfa isteklerinde hangi kullanıcının giriş yaptığını anlayabiliriz.


# 17. Session'a Veri Yazmak

In [ ]:
from flask import session

def ornek():
    session["user_id"] = 5


# 18. Session'dan Veri Okumak

In [ ]:
user_id = session.get("user_id")


`get()` kullanmak anahtar yokken hata oluşmasını önler.


# 19. Session'ı Temizlemek

Çıkış yaparken:


In [ ]:
session.clear()


Bu işlem mevcut session içeriğini temizler.


# 20. Kullanıcı Kayıt Sayfası

Kullanıcı kayıt formumuz:

```html
<form method="post">

    <label>Kullanıcı Adı</label>
    <input
        type="text"
        name="kullanici_adi"
        required
    >

    <label>Parola</label>
    <input
        type="password"
        name="parola"
        required
    >

    <label>Parola Tekrar</label>
    <input
        type="password"
        name="parola_tekrar"
        required
    >

    <button type="submit">
        Kayıt Ol
    </button>

</form>
```


# 21. Kullanıcı Adı Doğrulama

Kullanıcı adı:

- boş olmamalı,
- gereksiz boşluklar temizlenmeli,
- veritabanında benzersiz olmalıdır.

Örnek:


In [ ]:
kullanici_adi = request.form.get(
    "kullanici_adi",
    ""
).strip()


# 22. Parola Tekrar Kontrolü

In [ ]:
parola = request.form.get("parola", "")
parola_tekrar = request.form.get(
    "parola_tekrar",
    ""
)

if parola != parola_tekrar:
    print("Parolalar eşleşmiyor.")


# 23. Basit Parola Kuralları

Eğitim projemizde örnek olarak:

- en az 8 karakter,
- boş olmama

kontrolü uygulayabiliriz.

Gerçek uygulamalarda parola politikası sistemin güvenlik gereksinimlerine göre belirlenmelidir.


# 24. Kayıt Route'u

In [ ]:
@app.route("/kayit", methods=["GET", "POST"])
def kayit():
    if request.method == "POST":
        kullanici_adi = request.form.get(
            "kullanici_adi",
            ""
        ).strip()

        parola = request.form.get(
            "parola",
            ""
        )

        parola_tekrar = request.form.get(
            "parola_tekrar",
            ""
        )

        hata = None

        if not kullanici_adi:
            hata = "Kullanıcı adı gereklidir."

        elif len(parola) < 8:
            hata = "Parola en az 8 karakter olmalıdır."

        elif parola != parola_tekrar:
            hata = "Parolalar eşleşmiyor."

        if hata is None:
            try:
                db = get_db()

                db.execute("""
                INSERT INTO Kullanicilar (
                    KullaniciAdi,
                    ParolaHash,
                    Rol
                )
                VALUES (?, ?, ?)
                """, (
                    kullanici_adi,
                    generate_password_hash(parola),
                    "kullanici"
                ))

                db.commit()

            except sqlite3.IntegrityError:
                hata = "Bu kullanıcı adı zaten kayıtlı."

            else:
                flash("Kayıt tamamlandı. Giriş yapabilirsiniz.")

                return redirect(
                    url_for("giris")
                )

        flash(hata)

    return render_template("kayit.html")


# 25. Kayıt İşlemindeki Önemli Nokta

Veritabanına:

```python
parola
```

değil:

```python
generate_password_hash(parola)
```

sonucu yazılır.

Düz metin parola saklanmaz.


# 26. `UNIQUE` ve `IntegrityError`

Tabloda:

```sql
KullaniciAdi TEXT UNIQUE
```

bulunduğu için aynı kullanıcı adı ikinci kez eklenemez.

Bu durumda SQLite:

```python
sqlite3.IntegrityError
```

oluşturabilir.

Kullanıcıya teknik hata yerine anlaşılır mesaj gösteriyoruz.


# 27. Giriş Formu

`templates/giris.html`:

```html
{% extends "base.html" %}

{% block content %}

<h2>Giriş Yap</h2>

<form method="post">

    <p>
        <label>Kullanıcı Adı</label>

        <input
            type="text"
            name="kullanici_adi"
            required
        >
    </p>

    <p>
        <label>Parola</label>

        <input
            type="password"
            name="parola"
            required
        >
    </p>

    <button type="submit">
        Giriş Yap
    </button>

</form>

{% endblock %}
```


# 28. Kullanıcıyı Veritabanında Bulmak

In [ ]:
kullanici = get_db().execute(
    '''
    SELECT *
    FROM Kullanicilar
    WHERE KullaniciAdi = ?
    ''',
    ("ali",)
).fetchone()


# 29. Parola Kontrolü

Kullanıcı varsa:


In [ ]:
if kullanici is not None:
    dogru_mu = check_password_hash(
        kullanici["ParolaHash"],
        "GirilenParola"
    )

    print(dogru_mu)


# 30. Giriş Route'u

In [ ]:
@app.route("/giris", methods=["GET", "POST"])
def giris():
    if request.method == "POST":
        kullanici_adi = request.form.get(
            "kullanici_adi",
            ""
        ).strip()

        parola = request.form.get(
            "parola",
            ""
        )

        kullanici = get_db().execute("""
        SELECT *
        FROM Kullanicilar
        WHERE KullaniciAdi = ?
        """, (
            kullanici_adi,
        )).fetchone()

        hata = None

        if kullanici is None:
            hata = "Kullanıcı adı veya parola hatalı."

        elif not check_password_hash(
            kullanici["ParolaHash"],
            parola
        ):
            hata = "Kullanıcı adı veya parola hatalı."

        if hata is None:
            session.clear()
            session["user_id"] = kullanici["Id"]

            return redirect(
                url_for("ana_sayfa")
            )

        flash(hata)

    return render_template("giris.html")


# 31. Neden Hata Mesajını Genel Tuttuk?

Şu iki ayrı mesaj yerine:

```text
Bu kullanıcı yok.
Parola yanlış.
```

tek mesaj:

```text
Kullanıcı adı veya parola hatalı.
```

kullanmak, giriş ekranının gereksiz kullanıcı bilgisi sızdırmasını azaltır.


# 32. Giriş Başarılı Olduğunda

```python
session.clear()
session["user_id"] = kullanici["Id"]
```

uygulanır.

Önce eski session verisi temizlenir.

Ardından giriş yapan kullanıcının ID'si session'a yazılır.


# 33. Kullanıcı Bilgisini Her İstek Öncesi Yüklemek

Session'da yalnızca kullanıcı ID'sini tutuyoruz.

Her HTTP isteği öncesinde bu ID üzerinden kullanıcıyı veritabanından okuyabiliriz.


# 34. `before_request`

In [ ]:
@app.before_request
def kullaniciyi_yukle():
    user_id = session.get("user_id")

    if user_id is None:
        g.user = None

    else:
        g.user = get_db().execute(
            '''
            SELECT *
            FROM Kullanicilar
            WHERE Id = ?
            ''',
            (user_id,)
        ).fetchone()


Böylece route ve template'lerde:

```python
g.user
```

üzerinden giriş yapan kullanıcıya ulaşabiliriz.


# 35. `g` Nesnesi

`g`, bir HTTP isteği boyunca kullanılabilecek verileri tutmak için uygundur.

Örneğin:

```python
g.user
```

mevcut kullanıcıyı,

```python
g.db
```

veritabanı bağlantısını

tutabilir.

`g` verileri farklı istekler arasında kalıcı saklama alanı değildir.


# 36. Template'te Giriş Durumunu Göstermek

`base.html`:

```html
<nav>

{% if g.user %}

    <span>
        {{ g.user["KullaniciAdi"] }}
    </span>

    <a href="{{ url_for('profil') }}">
        Profil
    </a>

    <a href="{{ url_for('cikis') }}">
        Çıkış
    </a>

{% else %}

    <a href="{{ url_for('kayit') }}">
        Kayıt Ol
    </a>

    <a href="{{ url_for('giris') }}">
        Giriş
    </a>

{% endif %}

</nav>
```


# 37. Çıkış Route'u

In [ ]:
@app.route("/cikis")
def cikis():
    session.clear()

    flash("Oturum kapatıldı.")

    return redirect(
        url_for("ana_sayfa")
    )


Çıkış işleminin temel mantığı session içindeki kullanıcı bilgisini kaldırmaktır.


# 38. Giriş Gerektiren Sayfalar

Bazı sayfalara yalnızca giriş yapan kullanıcılar erişebilmelidir.

Örneğin:

```text
/profil
/ogrenci-ekle
/yonetim
```

Her route içinde tekrar tekrar giriş kontrolü yapmak yerine decorator kullanabiliriz.


# 39. `login_required` Decorator

Daha önce fonksiyon konularında decorator kavramını ayrıntılı işlemediysek burada kullanım mantığını görelim.


In [ ]:
def login_required(view):
    @functools.wraps(view)
    def wrapped_view(**kwargs):

        if g.user is None:
            flash("Bu sayfaya erişmek için giriş yapmalısınız.")

            return redirect(
                url_for("giris")
            )

        return view(**kwargs)

    return wrapped_view


# 40. Decorator'ı Kullanmak

In [ ]:
@app.route("/profil")
@login_required
def profil():
    return render_template(
        "profil.html"
    )


Artık giriş yapmamış kullanıcı:

```text
/profil
```

sayfasına giderse giriş ekranına yönlendirilir.


# 41. `profil.html`

```html
{% extends "base.html" %}

{% block content %}

<h2>Profil</h2>

<p>
    Kullanıcı adı:
    {{ g.user["KullaniciAdi"] }}
</p>

<p>
    Rol:
    {{ g.user["Rol"] }}
</p>

{% endblock %}
```


# 42. CRUD Route'larını Koruma

Önceki derste öğrenci ekleme route'umuz:

```python
@app.route("/ogrenci-ekle", methods=["GET", "POST"])
def ogrenci_ekle():
```

şeklindeydi.

Artık:

```python
@app.route("/ogrenci-ekle", methods=["GET", "POST"])
@login_required
def ogrenci_ekle():
```

yapabiliriz.


# 43. Yetkilendirme: Yönetici Rolü

Giriş yapmak tek başına her işlem için yeterli olmayabilir.

Örneğin:

- normal kullanıcı öğrenci listesini görebilir,
- yönetici kayıt silebilir.

Bunun için `Rol` alanını kullanabiliriz.


# 44. Yönetici Kullanıcısı Oluşturmak

Eğitim amacıyla veritabanına yönetici ekleyen bir fonksiyon yazabiliriz.


In [ ]:
def yonetici_olustur(kullanici_adi, parola):
    db = get_db()

    try:
        db.execute("""
        INSERT INTO Kullanicilar (
            KullaniciAdi,
            ParolaHash,
            Rol
        )
        VALUES (?, ?, ?)
        """, (
            kullanici_adi,
            generate_password_hash(parola),
            "yonetici"
        ))

        db.commit()

    except sqlite3.IntegrityError:
        pass


Gerçek uygulamalarda ilk yönetici hesabının oluşturulması kontrollü bir kurulum süreciyle yapılmalıdır.

Web üzerindeki herkese yönetici rolü seçtiren kayıt formu oluşturmak doğru değildir.


# 45. `admin_required` Decorator

Yalnızca yönetici rolüne izin veren decorator:


In [ ]:
def admin_required(view):
    @functools.wraps(view)
    def wrapped_view(**kwargs):

        if g.user is None:
            flash("Önce giriş yapmalısınız.")

            return redirect(
                url_for("giris")
            )

        if g.user["Rol"] != "yonetici":
            flash("Bu işlem için yetkiniz yok.")

            return redirect(
                url_for("ana_sayfa")
            )

        return view(**kwargs)

    return wrapped_view


# 46. Yönetim Sayfası

In [ ]:
@app.route("/yonetim")
@admin_required
def yonetim():
    kullanicilar = get_db().execute(
        '''
        SELECT Id, KullaniciAdi, Rol
        FROM Kullanicilar
        ORDER BY KullaniciAdi
        '''
    ).fetchall()

    return render_template(
        "yonetim.html",
        kullanicilar=kullanicilar
    )


# 47. Silme İşlemini Yalnızca Yöneticiye Açmak

Önceki CRUD route'u:

```python
@app.post("/ogrenci/<int:ogrenci_id>/sil")
```

üzerine:


In [ ]:
@admin_required
def ornek_silme():
    pass


Gerçek route:

```python
@app.post("/ogrenci/<int:ogrenci_id>/sil")
@admin_required
def ogrenci_sil(ogrenci_id):
    ...
```

şeklinde olabilir.


# 48. Template'te Role Göre Menü

```html
{% if g.user %}

    <a href="{{ url_for('profil') }}">
        Profil
    </a>

    {% if g.user["Rol"] == "yonetici" %}

        <a href="{{ url_for('yonetim') }}">
            Yönetim
        </a>

    {% endif %}

{% endif %}
```

Ancak önemli nokta:

**Template'te bağlantıyı gizlemek gerçek güvenlik değildir.**

Route tarafında da mutlaka yetki kontrolü yapılmalıdır.


# 49. Kullanıcı Rolünü Formdan Neden Almıyoruz?

Kayıt formunda:

```text
Rol seç:
- kullanıcı
- yönetici
```

gibi bir alan oluşturursak herkes kendisini yönetici yapabilir.

Bu nedenle normal kayıt route'u rolü sunucu tarafında:

```python
"kullanici"
```

olarak belirler.


# 50. Session Cookie Güvenliği

Üretim ortamında HTTPS kullanırken session cookie için güvenlik seçenekleri düşünülebilir.

Örnek:


In [ ]:
app.config.update(
    SESSION_COOKIE_SECURE=True,
    SESSION_COOKIE_HTTPONLY=True,
    SESSION_COOKIE_SAMESITE="Lax"
)


Bu ayarlar özellikle gerçek internet ortamında HTTPS ile birlikte değerlendirilmelidir.

Yerel `http://127.0.0.1` geliştirmesinde `SESSION_COOKIE_SECURE=True` kullanılırsa tarayıcı cookie'yi HTTP üzerinden göndermeyeceği için giriş davranışı beklediğiniz gibi çalışmayabilir.


# 51. Session'da Ne Saklamalıyız?

Session'a gereksiz veri doldurmak yerine çoğunlukla küçük bir kimlik bilgisi saklamak yeterlidir.

Örneğin:

```python
session["user_id"] = kullanici["Id"]
```

Sonraki istekte gerçek kullanıcı bilgilerini veritabanından okuyabiliriz.


# 52. Session Bir Veritabanı Değildir

Session:

- kullanıcı hesabının kalıcı kaydı değildir,
- öğrenci veritabanının yerine geçmez,
- büyük veri saklama yeri değildir.

Kalıcı bilgiler SQLite'ta tutulur.

Session yalnızca kullanıcı oturumu gibi isteklere yayılan küçük bilgileri taşımak için kullanılır.


# 53. Kullanıcı Adı mı ID mi Session'da?

Şunu kullanmak yerine:

```python
session["username"] = "ali"
```

uygulamamızda:

```python
session["user_id"] = 5
```

kullanmayı tercih ediyoruz.

ID veritabanındaki kullanıcı kaydına doğrudan bağlanır.


# 54. Kullanıcı Silinmişse Ne Olur?

Session'da bir `user_id` olabilir fakat ilgili kayıt veritabanından silinmiş olabilir.

`before_request` fonksiyonumuzda:

```python
g.user = db.execute(...).fetchone()
```

sonucu bu durumda `None` olur.

Böylece kullanıcı giriş yapmamış kabul edilir.


# 55. Giriş Sonrası Kullanıcının Geldiği Sayfaya Dönme

Daha gelişmiş uygulamalarda kullanıcı korumalı bir sayfaya girmek isterken giriş ekranına yönlendirilirse, giriş sonrasında aynı sayfaya geri döndürülebilir.

Bu işlem `next` parametresi gibi yapılarla geliştirilebilir.

Ancak dışarıdan gelen yönlendirme adresleri doğrulanmadan kullanılmamalıdır.

Bu derste daha basit ve güvenli akış olarak kullanıcıyı ana sayfaya yönlendiriyoruz.


# 56. Flash Mesajlarını Kategorilere Ayırmak

Flask `flash()` mesajlarına kategori eklenebilir.

Örnek:

```python
flash(
    "Giriş başarılı.",
    "success"
)

flash(
    "Parola hatalı.",
    "error"
)
```

Template tarafında kategoriye göre farklı görünüm verilebilir.


# 57. Kategorili Flash Mesajlarını Göstermek

`base.html`:

```html
{% with mesajlar = get_flashed_messages(with_categories=true) %}

    {% for kategori, mesaj in mesajlar %}

        <div class="mesaj {{ kategori }}">
            {{ mesaj }}
        </div>

    {% endfor %}

{% endwith %}
```


# 58. Kayıt Formunda Kullanıcı Adını Korumak

Kayıt doğrulaması hata verirse kullanıcı adını tekrar yazdırmak istemeyebiliriz.

HTML:

```html
<input
    type="text"
    name="kullanici_adi"
    value="{{ request.form.get('kullanici_adi', '') }}"
>
```

Parola alanını ise tekrar doldurmamak daha uygundur.


# 59. Login Formunda Parolayı Geri Yazmamak

Şu kullanım yapılmamalıdır:

```html
<input
    type="password"
    value="{{ request.form.get('parola') }}"
>
```

Parola hatası sonrası parola alanını boş bırakmak daha doğru bir yaklaşımdır.


# 60. Kullanıcı Tablosunu Öğrenci Tablosundan Ayırmak

İyi tasarım:

```text
Kullanicilar
-----------
Id
KullaniciAdi
ParolaHash
Rol

Ogrenciler
----------
Id
Isim
Sinif
PythonPuani
MatematikPuani
```

Kullanıcı hesabı ve öğrenci kaydı farklı kavramlardır.

Gerekirse ileride aralarında ilişki kurulabilir.


# 61. Yetkilendirme Senaryosu

Örnek uygulama kuralları:

### Ziyaretçi

- ana sayfayı görür
- giriş yapabilir
- kayıt olabilir

### Kullanıcı

- öğrenci listesini görür
- profilini görür

### Yönetici

- öğrenci ekler
- öğrenci günceller
- öğrenci siler
- kullanıcı listesini görür

Bu kurallar kodla açık biçimde uygulanmalıdır.


# 62. Endpoint Koruma Tablosu

| Route | Ziyaretçi | Kullanıcı | Yönetici |
|---|---|---|---|
| `/` | Evet | Evet | Evet |
| `/giris` | Evet | Evet | Evet |
| `/kayit` | Evet | Evet | Evet |
| `/profil` | Hayır | Evet | Evet |
| `/ogrenciler` | Hayır | Evet | Evet |
| `/ogrenci-ekle` | Hayır | Hayır | Evet |
| `/ogrenci/<id>/sil` | Hayır | Hayır | Evet |

Proje başlamadan böyle bir tablo oluşturmak yetkilendirme tasarımını kolaylaştırır.


# 63. Tam Veritabanı Hazırlama Örneği

In [ ]:
def veritabani_hazirla():
    with sqlite3.connect(DATABASE) as db:

        db.execute("""
        CREATE TABLE IF NOT EXISTS Kullanicilar (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            KullaniciAdi TEXT UNIQUE NOT NULL,
            ParolaHash TEXT NOT NULL,
            Rol TEXT NOT NULL DEFAULT 'kullanici'
        )
        """)

        db.execute("""
        CREATE TABLE IF NOT EXISTS Ogrenciler (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            Isim TEXT NOT NULL,
            Sinif INTEGER NOT NULL,
            PythonPuani INTEGER NOT NULL,
            MatematikPuani INTEGER NOT NULL
        )
        """)


# 64. Tam Giriş Sistemi İçin Proje Yapısı

```text
ogrenci_web/
│
├── app.py
├── instance/
│   └── uygulama.db
│
├── templates/
│   ├── base.html
│   ├── index.html
│   ├── kayit.html
│   ├── giris.html
│   ├── profil.html
│   ├── yonetim.html
│   ├── ogrenciler.html
│   ├── ogrenci_ekle.html
│   └── ogrenci_duzenle.html
│
└── static/
    └── css/
        └── style.css
```

Uygulama büyüdükçe auth işlemlerini ayrı modüle veya Blueprint'e ayırabiliriz.


# 65. Blueprint'e Hazırlık

Daha büyük Flask projelerinde:

```text
auth.py
ogrenciler.py
db.py
```

gibi dosyalar oluşturabiliriz.

Örneğin authentication route'ları:

```text
/auth/kayit
/auth/giris
/auth/cikis
```

gibi bir Blueprint içinde toplanabilir.

Bu ders temel yapıyı tek uygulamada anlamaya odaklanır.


# 66. Tam `login_required` Örneği

In [ ]:
def login_required(view):
    @functools.wraps(view)
    def wrapped_view(**kwargs):

        if g.user is None:
            flash(
                "Bu sayfa için giriş yapmalısınız.",
                "warning"
            )

            return redirect(
                url_for("giris")
            )

        return view(**kwargs)

    return wrapped_view


# 67. Tam `admin_required` Örneği

In [ ]:
def admin_required(view):
    @functools.wraps(view)
    def wrapped_view(**kwargs):

        if g.user is None:
            flash(
                "Önce giriş yapmalısınız.",
                "warning"
            )

            return redirect(
                url_for("giris")
            )

        if g.user["Rol"] != "yonetici":
            flash(
                "Bu işlem için yetkiniz yok.",
                "error"
            )

            return redirect(
                url_for("ana_sayfa")
            )

        return view(**kwargs)

    return wrapped_view


# 68. Tam `before_request` Örneği

In [ ]:
@app.before_request
def kullaniciyi_yukle():
    user_id = session.get("user_id")

    if user_id is None:
        g.user = None
        return

    g.user = get_db().execute(
        '''
        SELECT
            Id,
            KullaniciAdi,
            Rol
        FROM Kullanicilar
        WHERE Id = ?
        ''',
        (user_id,)
    ).fetchone()


Parola hash değerini her template için `g.user` içine taşımamız gerekmez.

Sorguda yalnızca ihtiyaç duyduğumuz alanları seçebiliriz.


# 69. Yönetici Route Örneği

In [ ]:
@app.route("/yonetim")
@admin_required
def yonetim():
    kullanicilar = get_db().execute(
        '''
        SELECT
            Id,
            KullaniciAdi,
            Rol
        FROM Kullanicilar
        ORDER BY KullaniciAdi
        '''
    ).fetchall()

    return render_template(
        "yonetim.html",
        kullanicilar=kullanicilar
    )


# 70. Yönetim Template Örneği

```html
{% extends "base.html" %}

{% block content %}

<h2>Kullanıcı Yönetimi</h2>

<table>

    <tr>
        <th>ID</th>
        <th>Kullanıcı Adı</th>
        <th>Rol</th>
    </tr>

    {% for kullanici in kullanicilar %}

    <tr>
        <td>{{ kullanici["Id"] }}</td>
        <td>{{ kullanici["KullaniciAdi"] }}</td>
        <td>{{ kullanici["Rol"] }}</td>
    </tr>

    {% endfor %}

</table>

{% endblock %}
```


# 71. Kullanıcı Rolünü Güncelleme

İleri geliştirme olarak yönetici bir kullanıcının rolünü değiştirebilir.

SQL:

```sql
UPDATE Kullanicilar
SET Rol = ?
WHERE Id = ?
```

Ancak:

- herkesin rol değiştirmesine izin verilmemeli,
- yetki yükseltme işlemleri yönetici kontrolünde olmalı,
- yönetici kendi kritik yetkisini yanlışlıkla kaybetmeye karşı korunabilir.


# 72. Session Güvenliğinin Temeli

Flask'ın yerleşik session sistemi istemci tarafındaki imzalı cookie mekanizmasını kullanır.

Bu nedenle:

- `SECRET_KEY` güçlü olmalıdır,
- session içine hassas düz metin parola yazılmamalıdır,
- cookie içeriği gizli veritabanı olarak düşünülmemelidir.


# 73. Session İçine Parola Yazmayın

Yanlış:

```python
session["parola"] = parola
```

Doğru yaklaşım:

```python
session["user_id"] = kullanici["Id"]
```

Parola yalnızca giriş doğrulama sırasında kullanılır ve session'a kaydedilmez.


# 74. Logout Sonrası Davranış

Çıkış yaptıktan sonra:

```python
session.clear()
```

uygulanır.

Sonraki istekte:

```python
session.get("user_id")
```

`None` döndürür.

`g.user` da `None` olur.

Korunan route'lar kullanıcıyı giriş sayfasına yönlendirir.


# 75. Güvenlik: Debug Modu

Geliştirme sırasında:

```text
flask --app app run --debug
```

kullanışlıdır.

Ancak debug modu gerçek internete açık üretim ortamında kullanılmamalıdır.

Üretimde hata ayıklayıcı kapalı olmalıdır.


# 76. Güvenlik: CSRF Farkındalığı

Kullanıcı oturumu bulunan web uygulamalarında form işlemleri için CSRF koruması önemlidir.

Örneğin:

- kullanıcı silme,
- rol değiştirme,
- profil güncelleme

gibi POST işlemleri yalnızca doğru formdan geldiği doğrulanacak şekilde korunmalıdır.

Temel Flask çekirdeği form alanlarına otomatik CSRF token eklemez. Daha gelişmiş projelerde uygun bir CSRF koruma mekanizması veya Flask eklentisi kullanılmalıdır.

Bu ders authentication ve authorization mantığına odaklandığı için CSRF uygulamasını sonraki güvenlik aşamasına bırakıyoruz.


# 77. Güvenlik: SQL Parametreleri

Authentication sorgularında da parametreli SQL kullanmaya devam ediyoruz.

Doğru:

```python
db.execute(
    "SELECT * FROM Kullanicilar WHERE KullaniciAdi = ?",
    (kullanici_adi,)
)
```

Yanlış:

```python
"SELECT ... WHERE KullaniciAdi = '" + kullanici_adi + "'"
```


# 78. Güvenlik: Parola Hash Algoritmasını Elle Yazmayın

Kendi:

- şifreleme algoritmanızı,
- hash sisteminizi,
- salt mekanizmanızı

tasarlamak yerine güvenilir ve güncel kütüphane araçlarını kullanmak gerekir.

Bu nedenle Werkzeug'un parola yardımcı fonksiyonlarını kullanıyoruz.


# 79. Kullanıcı Giriş Sistemi Akışı

```text
Kayıt
↓
Parolayı hashle
↓
Hash'i SQLite'a yaz

Giriş
↓
Kullanıcıyı bul
↓
check_password_hash()
↓
session.clear()
↓
session["user_id"]

Her İstek
↓
before_request
↓
g.user

Korunan Route
↓
login_required / admin_required
```


# 80. Tam Uygulama İçinde Giriş ve CRUD

Artık önceki dersimizdeki öğrenci CRUD route'larını şu şekilde koruyabiliriz:

```python
@app.route("/ogrenciler")
@login_required
def ogrenci_listesi():
    ...

@app.route("/ogrenci-ekle", methods=["GET", "POST"])
@admin_required
def ogrenci_ekle():
    ...

@app.route(
    "/ogrenci/<int:ogrenci_id>/duzenle",
    methods=["GET", "POST"]
)
@admin_required
def ogrenci_duzenle(ogrenci_id):
    ...

@app.post("/ogrenci/<int:ogrenci_id>/sil")
@admin_required
def ogrenci_sil(ogrenci_id):
    ...
```

Bu yapı kimlik doğrulama ile CRUD sistemimizi birleştirir.


# 81. Kullanıcı Deneyimi

Giriş sistemi yalnızca teknik olarak çalışmamalı, kullanıcıya anlaşılır davranmalıdır.

Örneğin:

- kayıt tamamlandı mesajı,
- hatalı giriş mesajı,
- çıkış yapıldı mesajı,
- yetkiniz yok mesajı,
- giriş yapmanız gerekiyor mesajı

gösterilmelidir.


# 82. Sık Yapılan Hatalar

### Parolayı düz metin saklamak

Yanlış.

### Secret key'i GitHub'a yüklemek

Kaçınılmalıdır.

### Kullanıcı rolünü kayıt formundan almak

Yetki yükseltme açığına neden olabilir.

### Yalnızca menü bağlantısını gizlemek

Route tarafında yetki kontrolü yoksa yeterli değildir.

### Session'a parola koymak

Yapılmamalıdır.

### SQL sorgularını string birleştirmeyle oluşturmak

Parametreli sorgular kullanılmalıdır.

### Debug modunu üretimde açık bırakmak

Güvenli değildir.


# 83. Test Senaryoları

Uygulamayı şu durumlarda test edin:

1. yeni kullanıcı kayıt olabiliyor mu?
2. aynı kullanıcı adı tekrar kaydedilebiliyor mu?
3. yanlış parola ile giriş engelleniyor mu?
4. doğru parola ile giriş oluyor mu?
5. giriş sonrası `g.user` doluyor mu?
6. çıkış sonrası profil sayfası engelleniyor mu?
7. normal kullanıcı yönetim sayfasına girebiliyor mu?
8. yönetici yönetim sayfasına girebiliyor mu?
9. giriş yapmamış kullanıcı CRUD işlemi yapabiliyor mu?
10. silme route'u gerçekten yetki kontrolü yapıyor mu?


# 84. Manuel Test Tablosu

| Test | Beklenen Sonuç |
|---|---|
| Boş kullanıcı adı | Kayıt engellenir |
| Kısa parola | Kayıt engellenir |
| Parolalar farklı | Kayıt engellenir |
| Aynı kullanıcı adı | Hata mesajı |
| Yanlış parola | Giriş engellenir |
| Doğru parola | Session oluşur |
| Çıkış | Session temizlenir |
| Ziyaretçi profil | Girişe yönlendirilir |
| Kullanıcı yönetim | Yetki reddedilir |
| Yönetici yönetim | Sayfa açılır |

Bu tür test tabloları proje geliştirmede çok yararlıdır.


# 85. Ders Özeti

Bu derste:

- authentication,
- authorization,
- kullanıcı tablosu,
- `generate_password_hash()`,
- `check_password_hash()`,
- secret key,
- ortam değişkeni,
- session,
- `session.clear()`,
- `session["user_id"]`,
- `before_request`,
- `g.user`,
- kayıt olma,
- giriş yapma,
- çıkış yapma,
- `login_required`,
- `admin_required`,
- roller,
- yönetici sayfası,
- flash mesajları,
- cookie güvenlik ayarları,
- SQL Injection farkındalığı,
- CSRF farkındalığı,
- parola güvenliği,
- route yetkilendirme,
- authentication testleri

konularını öğrendik.


# 86. Mini Uygulamalar

1. `Kullanicilar` tablosunu oluşturun.
2. Kullanıcı adı alanını `UNIQUE` yapın.
3. Kayıt formu hazırlayın.
4. Parola tekrar alanı ekleyin.
5. En az 8 karakter kontrolü yapın.
6. Parolayı hashleyerek kaydedin.
7. Aynı kullanıcı adına hata mesajı gösterin.
8. Giriş formu hazırlayın.
9. Yanlış parola ile girişi engelleyin.
10. Doğru girişte `session["user_id"]` oluşturun.
11. `before_request` ile `g.user` yükleyin.
12. Çıkış route'u oluşturun.
13. Profil sayfasını `login_required` ile koruyun.
14. Öğrenci listesi sayfasını yalnızca giriş yapanlara açın.
15. `Rol` alanı ekleyin.
16. Yönetici kullanıcısı oluşturun.
17. `admin_required` decorator yazın.
18. Silme route'unu yalnızca yöneticiye açın.
19. Menüde role göre Yönetim bağlantısı gösterin.
20. Yetkisiz kullanıcıya flash mesajı gösterin.
21. Flash mesajlarına kategori ekleyin.
22. Cookie güvenlik ayarlarını araştırıp geliştirme ve üretim farkını açıklayın.
23. Secret key'i ortam değişkeninden okuyun.
24. Giriş sistemi için en az 10 test senaryosu hazırlayın.
25. Önceki Flask CRUD uygulamanıza tam kullanıcı giriş sistemi ekleyin.


# 87. Proje Görevi

Önceki **BİLSEM Proje Takip Web Uygulamasını** kullanıcı giriş sistemiyle geliştirin.

Kullanıcı tablosu:

```text
Id
KullaniciAdi
ParolaHash
Rol
```

Roller:

```text
kullanici
yonetici
```

### Kullanıcı

- giriş yapabilir,
- projeleri görüntüleyebilir,
- profilini görüntüleyebilir.

### Yönetici

- proje ekleyebilir,
- proje düzenleyebilir,
- proje silebilir,
- kullanıcı listesini görüntüleyebilir.

Projede en az:

- kullanıcı kayıt,
- giriş,
- çıkış,
- parola hash,
- session,
- `g.user`,
- `login_required`,
- `admin_required`,
- flash mesajları,
- rol kontrolü,
- SQLite,
- CRUD

bulunsun.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin şu güvenli web uygulaması akışını kurabilmesi hedeflenmektedir:

**Kayıt**

↓

**Parola Hash**

↓

**SQLite Kullanıcı Kaydı**

↓

**Giriş**

↓

**Parola Kontrolü**

↓

**Session**

↓

**g.user**

↓

**Kimlik Doğrulama**

↓

**Rol Kontrolü**

↓

**Yetkilendirilmiş CRUD İşlemleri**

Bu noktada uygulamamız artık yalnızca veri ekleyen bir web sayfası değildir.

Kullanıcı kimliği ve yetkisine göre davranan gerçek bir **çok kullanıcılı web uygulamasının temel yapısına** dönüşmüştür.
